In [8]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
data_path = '/content/drive/MyDrive/cu_ai_grant/data'
metrics_path = '/content/drive/MyDrive/cu_ai_grant/metrics'
BASE = "Qwen/Qwen3-4B-Instruct-2507"

In [22]:
!pip -q install -U "transformers==4.55.2" "trl==0.19.1" "peft==0.16.0" bitsandbytes datasets scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires hugging

In [36]:
import json, torch, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
import pickle, warnings
from scipy.sparse import hstack
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
warnings.filterwarnings("ignore")

set_seed(42)

# Основная часть кода

Считываем данные

In [30]:
def load_jsonl(p): return [json.loads(l) for l in open(p)]

kid_adult = load_jsonl(f"{data_path}/kid_adult.jsonl")
good_bad = load_jsonl(f"{data_path}/good_bad.jsonl")
test_style = load_jsonl(f"{data_path}/public_test_style.jsonl")
test_quality = load_jsonl(f"{data_path}/public_test_quality.jsonl")

d = pickle.load(open(f"{metrics_path}/style_clf.pkl", "rb"))
style_clf, style_vecs = d["clf"], d["vecs"]

In [34]:
def p_simple(texts):
    X = hstack([v.transform(texts) for v in style_vecs])
    return style_clf.predict_proba(X)[:, 1]

print(p_simple([r["kid"] for r in test_style]).mean(), p_simple([r["adult"] for r in test_style]).mean())

0.9747509734539157 0.01841233282530796


In [38]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")

sft_data = Dataset.from_list([{"messages": [
    {"role": "user", "content": r["prompt"]},
    {"role": "assistant", "content": r["kid"]}]} for r in kid_adult])

# короче ранг 16 экспериментальным путем, большой слишком конечно я не ставил потому что ждать лень, alpha=2r база, слои по максимуму захватил
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

args = SFTConfig(output_dir="sft", num_train_epochs=1, per_device_train_batch_size=2,
                 gradient_accumulation_steps=8, learning_rate=2e-4, lr_scheduler_type="cosine",
                 warmup_ratio=0.03, max_length=1024, fp16=True, logging_steps=10,
                 report_to="none", seed=42)

trainer = SFTTrainer(model=model, args=args, train_dataset=sft_data,
                     processing_class=tok, peft_config=lora)
trainer.train()
trainer.save_model("sft")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Step,Training Loss
10,2.123000
20,1.440900
30,1.333800
40,1.222600
50,1.216600
60,1.178300
70,1.189800
80,1.170000
90,1.193100


In [41]:
def generate(model, prompts, max_new=256):
    model.eval()
    outs = []
    for p in prompts:
        ids = tok.apply_chat_template([{"role": "user", "content": p}],
                                      add_generation_prompt=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=max_new, do_sample=False)
        outs.append(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
    return outs

gens = generate(trainer.model, [r["prompt"] for r in test_style])
m1 = float(p_simple(gens).mean())
print("mean P_simple =", round(m1, 4))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

mean P_simple = 0.9787


In [42]:
!git add

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ai_red.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/
	metrics/
	sft/

no changes added to commit (use "git add" and/or "git commit -a")
